In [0]:
# Multiple ways to read a Data

'''
1. Postgres & MySQL using JDBC 
2. S3
3. Snowflake
4. DBFS 
5. ADLS
6. RDS
'''

In [0]:
# -----------------------------------------------
# Read data from PostgreSQL database using JDBC
# -----------------------------------------------


# Modern way: keep only connection details in URL
# credentials will be passed separately using properties
jdbc_url = "jdbc:postgresql://razz-db-therazz007-878c.b.aivencloud.com:22917/defaultdb"


# JDBC connection properties
# driver  -> JDBC driver required for PostgreSQL
# user    -> database username
# password-> database password
properties = {
     "driver": "org.postgresql.Driver",   
    "user": "avnadmin",
    "password": "<DBPASSWORD>"
}


# Read data from PostgreSQL table "orders"
# Spark internally runs: SELECT * FROM orders
# and converts the result into a Spark DataFrame
df = spark.read.jdbc(jdbc_url, table="orders", properties=properties)


# Display the first 20 rows of the dataframe
df.show()

In [0]:
# -----------------------------------------------
# Read data using format("jdbc") and option()
# -----------------------------------------------

df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://razz-db-therazz007-878c.b.aivencloud.com:22917/defaultdb") \
    .option("dbtable", "orders") \
    .option("user", "avnadmin") \
    .option("password", "<DBPASSWORD>") \
    .option("driver", "org.postgresql.Driver") \
    .load()


# Show data
df.show()

In [0]:
# Example: Reading a public Parquet file from the NYC Taxi dataset
path = "s3a://nyc-tlc/trip data/yellow_tripdata_2023-01.parquet"

df = spark.read.parquet(path)
df.show(5)

In [0]:
# Run SQL query on Delta table
df = spark.sql("""
SELECT city_name, country_code, cloud_base_height
FROM samples.accuweather.forecast_hourly_imperial
LIMIT 10
""")

# Show result
df.show()

## Read CSV Data from DBFS Volume



In [0]:
# how to read csv data from DBFS volume 


path = r'/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv'


df = spark.read.csv(path,header=True,inferSchema=True)
df.show(5,truncate=False)



In [0]:

df = spark.read.csv("dbfs:/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv",header=True,inferSchema=True)
df.show(5,truncate=False)


In [0]:
# another ways to read csv 


options = {
    "header": "true", 
    "inferSchema": "true",
    "delimiter": ",",
    "quote": "\"",
    "escape": "\"",
    "multiLine": "true",

}
df  = spark.read.format("csv").options(**options).option("path","dbfs:/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv").load()
df.show(5,truncate=False)

## Read JSON Data


In [0]:
jsonPath = "/Volumes/workspace/default/products-internal/products.json"


properties = {
    "multiline": "true",
 
    "inferSchema": "true",
    "dateFormat": "yyyy-MM-dd'T'HH:mm:ss",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss",
    "locale": "en-US",
    "encoding": "UTF-8",
}

df = spark.read.format("json").options(**properties).option("path",jsonPath).load()
df.show(5,truncate=False)



df.printSchema()


In [0]:
jsonPath = "/Volumes/workspace/default/products-internal/products.json"

df = spark.read \
    .option("multiline", "true") \
    .json(jsonPath)

df.show() 

In [0]:
from pyspark.sql.functions import explode

df_products = df.select(
    explode("products.data.items").alias("product")
)
df_products.show(1)

df_final = df_products.select("product.*")

df_final.show(1)

In Spark (especially **PySpark**), reading data is a core operation and there are **multiple patterns depending on data source, format, and use case**.

---

# 🔹 1. Core Entry Points

Spark provides two main APIs:

### ✅ DataFrame API (most used)

```python
df = spark.read.format("csv").load("path")
```

### ✅ Spark SQL

```python
df = spark.sql("SELECT * FROM table_name")
```

---

# 🔹 2. Reading by File Format

## 📁 2.1 CSV

```python
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/path/file.csv")
```

👉 Production tip:

* Avoid `inferSchema` in large datasets → define schema explicitly.

---

## 📁 2.2 JSON

```python
df = spark.read \
    .option("multiline", "true") \
    .json("/path/file.json")
```

With schema:

```python
df = spark.read.schema(schema).json("/path/file.json")
```

---

## 📁 2.3 Parquet (Most optimized)

```python
df = spark.read.parquet("/path/file.parquet")
```

👉 Best for:

* Column pruning
* Predicate pushdown
* Compression

---

## 📁 2.4 ORC

```python
df = spark.read.orc("/path/file.orc")
```

---

## 📁 2.5 Delta (Databricks / Lakehouse)

```python
df = spark.read.format("delta").load("/path/delta-table")
```

Or:

```python
df = spark.read.table("table_name")
```

---

# 🔹 3. Generic Format Loader

```python
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/path/file.csv")
```

👉 Use when:

* Writing reusable pipelines
* Dynamic formats

---

# 🔹 4. Reading from Databases (JDBC)

```python
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://host:5432/db") \
    .option("dbtable", "users") \
    .option("user", "username") \
    .option("password", "password") \
    .load()
```

👉 Production best practices:

* Use partitioning:

```python
.option("partitionColumn", "id") \
.option("lowerBound", 1) \
.option("upperBound", 100000) \
.option("numPartitions", 10)
```

---

# 🔹 5. Reading from Hive / Catalog Tables

```python
df = spark.read.table("db_name.table_name")
```

or

```python
df = spark.sql("SELECT * FROM db_name.table_name")
```

---

# 🔹 6. Reading Streaming Data

```python
df = spark.readStream \
    .format("json") \
    .load("/stream/path")
```

👉 Used for:

* Kafka
* File streams
* Real-time pipelines

---

# 🔹 7. Reading from RDD (Low-level)

```python
rdd = spark.sparkContext.textFile("/path/file.txt")
df = rdd.toDF()
```

👉 Rarely used now (legacy / custom transformations)

---

# 🔹 8. Reading In-Memory Data

### From Python list

```python
data = [("Alice", 25), ("Bob", 30)]
df = spark.createDataFrame(data, ["name", "age"])
```

---

### From Pandas

```python
import pandas as pd

pdf = pd.DataFrame({"name": ["A", "B"], "age": [10, 20]})
df = spark.createDataFrame(pdf)
```

---

# 🔹 9. Advanced Patterns (Important in Real Projects)

## 🔸 9.1 Schema Enforcement

```python
spark.read.schema(schema).csv(path)
```

✔ avoids runtime issues
✔ faster execution

---

## 🔸 9.2 Partitioned Data Reading

```python
df = spark.read.parquet("/data/year=2025/month=03/")
```

👉 Spark automatically detects partitions

---

## 🔸 9.3 Reading Multiple Files

```python
df = spark.read.csv("/data/*.csv")
```

---

## 🔸 9.4 Recursive File Read

```python
spark.read.option("recursiveFileLookup", "true").csv("/data/")
```

---

## 🔸 9.5 Sampling (for large datasets)

```python
df = spark.read.option("samplingRatio", 0.1).json(path)
```

---

# 🔹 10. Key Engineering Takeaways

### ⚡ Choose format wisely

| Format  | Use Case             |
| ------- | -------------------- |
| CSV     | Raw ingestion        |
| JSON    | Semi-structured      |
| Parquet | Analytics (best)     |
| Delta   | Production lakehouse |

---

### ⚡ Performance Rules

* Always define schema in production
* Prefer Parquet/Delta over CSV/JSON
* Use partition pruning
* Use predicate pushdown

---


